## Data Source 

The dataset used in this project comes from the `rust-data` repository provided by the OpenSourceEconomics community. 

I initially identified this data source through the official `ruspy` documentation: 

https://ruspy.readthedocs.io/en/latest/index.html 

In the documentation, the authors suggest that users may use the functions in `zurcher-data` provided by the OpenSourceEconomics community. Following this recommendation, I located the corresponding GitHub repository: 

https://github.com/OpenSourceEconomics/rust-data 

The repository contains both the original Rust (1987) bus replacement data and the associated data-processing functions. 

Instead of manually parsing the original `.asc` files, I use the preprocessing pipeline provided in the repository to convert the raw records into a structured panel dataset suitable for dynamic discrete choice estimation and later neural-network-based extensions.

## Data Extraction and Cleaning


This project uses the original bus replacement data from Rust (1987), accessed through the `rust-data` repository provided by OpenSourceEconomics. 
The raw data are stored in the original format used for the replication of the Rust bus engine replacement model. Instead of manually parsing the raw `.asc` files, I use the provided data-processing functions to convert the raw records into a structured panel dataset. 
The original dataset contains monthly observations for individual buses. For each bus, the data record includes odometer readings over time and information about engine replacement events. 
Since the Rust model is a dynamic discrete choice model, the relevant empirical object is a panel dataset indexed by bus and time period. 

--- 
### Main Variables 
The cleaned dataset contains four main variables: 
- `state`: the discretized mileage state of the bus 
- `mileage`: the original odometer reading 
- `usage`: the change in discretized mileage state between periods - `decision`: the observed replacement decision 

where: 
- `0` = keep the current engine 
- `1` = replace the engine 

--- 
### Mileage Discretization 
The mileage variable is discretized using a bin size of 5,000 miles. 
This follows the logic of the dynamic programming model, where the continuous mileage variable must be represented on a finite state grid before value function iteration or likelihood-based estimation can be implemented. 

--- 
### Sample Selection 
For the main analysis, I focus on **bus group 4**. 
This group corresponds to one of the main GMC bus groups used in the Rust replacement model and provides a clean starting point for reproducing the classical benchmark before extending the model with neural-network-based methods. 
The processed dataset is indexed by `Bus_ID` and `period`, which allows the data to be interpreted as a panel structure suitable for dynamic discrete choice estimation.

In [64]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add the local rust-data repository to the Python path
sys.path.append("../data/rust-data")

from data.data_processing import data_reading, data_processing

In [65]:
# Convert the original raw files into intermediate group-level pickle files.
# This only needs to be run once, but it is kept here for reproducibility.
data_reading()  

In [66]:
# Process group 4 into the cleaned panel format used for estimation.
init_dict = {
    "groups": "group_4",
    "binsize": 5000
}

df_clean = data_processing(init_dict)

In [67]:
# Convert columns to numeric types.
df_clean["state"] = pd.to_numeric(df_clean["state"])
df_clean["mileage"] = pd.to_numeric(df_clean["mileage"])
df_clean["usage"] = pd.to_numeric(df_clean["usage"])
df_clean.head()

state  mileage  usage  decision
Bus_ID period                                 
5297   0           0   2353.0    NaN         0
       1           1   6299.0    1.0         0
       2           2  10479.0    1.0         0
       3           3  15201.0    1.0         0
       4           4  20326.0    1.0         0

In [68]:
df_clean.describe()

,state,mileage,usage,decision
count,4329.000000,4329.000000,4292.000000,4329.000000
mean,25.395934,129454.487179,0.620923,0.007623
std,18.940754,94712.232318,0.510948,0.086986
min,0.000000,5.000000,0.000000,0.000000
25%,9.000000,47911.000000,0.000000,0.000000
50%,21.000000,109232.000000,1.000000,0.000000
75%,40.000000,203623.000000,1.000000,0.000000
max,77.000000,387282.000000,2.000000,1.000000


In [69]:
df_clean["decision"].value_counts()

decision
0    4296
1      33
Name: count, dtype: int64